Johan Camilo Balanta Santacruz - 30000115778
Delio Fernando Palacios Viafara - 30000115782

# Modelo de Regresión Híbrido CNN-LSTM para la Predicción de Alquiler de Bicicletas

**Dataset:** [Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)  
**Tasa de muestreo e Instancias:** 17,379 registros con frecuencia horaria (años 2011–2012).  
**Objetivo del estudio:** Diseñar un modelo híbrido que combine Redes Convolucionales (CNN) para extracción de patrones locales y Redes LSTM para memoria temporal de largo plazo.

## Instalación e importaciones

In [ ]:
%pip install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


## Preprocesamiento y Feature Engineering Avanzado

Para mejorar la capacidad predictiva del modelo, se aplican técnicas avanzadas de ingeniería de características:

1. **Codificación Cíclica:** Las variables temporales (hora, día, mes) son circulares (la hora 23 está cerca de la 0). Se transforman a coordenadas polares (seno/coseno) para preservar esta continuidad.
2. **Lag Features (Retardos):** Se incluyen valores pasados del target (`cnt` en $t-1$, $t-24$, $t-168$) como inputs. Dado que el consumo de bicicletas tiene una fuerte autocorrelación (lo que pasó ayer a esta hora influye en hoy), estas variables son los predictores más potentes.
3. **Rolling Features (Estadísticas Móviles):**
    * **Media Móvil** en ventanas de 3h, 6h, 12h, 24h y 168h: capturan tendencias a corto, mediano y largo plazo.
    * **Valores extremos** (máximo/mínimo en 24h): detectan picos y valles recientes.
    * **Desviación Estándar** en 24h y 168h: miden la volatilidad del sistema.
4. **Features de interacción:** `temp × hum` como proxy de la sensación térmica percibida, que afecta directamente la decisión de alquilar.
5. **Log-Transform ($\log(x+1)$):** Los datos de conteo suelen tener una distribución sesgada a la derecha (muchos valores bajos, pocos picos muy altos). El logaritmo estabiliza la varianza (homocedasticidad) y facilita que el modelo entrene de manera más eficiente.

In [ ]:
dataset = fetch_ucirepo(id=275)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

df['dteday'] = pd.to_datetime(df['dteday'], errors='coerce')
df = df.dropna(subset=['dteday'])
df = df.sort_values('dteday').reset_index(drop=True)
df = df.drop_duplicates(subset=['dteday', 'hr']).reset_index(drop=True)

df['hour_sin']  = np.sin(2 * np.pi * df['hr'] / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hr'] / 24)
df['dow_sin']   = np.sin(2 * np.pi * df['weekday'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['weekday'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['mnth'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['mnth'] / 12)

df['cnt_lag_1']   = df['cnt'].shift(1)
df['cnt_lag_24']  = df['cnt'].shift(24)
df['cnt_lag_168'] = df['cnt'].shift(168)

df['cnt_roll_mean_3']   = df['cnt'].rolling(window=3,   min_periods=1).mean()
df['cnt_roll_mean_6']   = df['cnt'].rolling(window=6,   min_periods=1).mean()
df['cnt_roll_mean_12']  = df['cnt'].rolling(window=12,  min_periods=1).mean()
df['cnt_roll_mean_24']  = df['cnt'].rolling(window=24,  min_periods=1).mean()
df['cnt_roll_mean_168'] = df['cnt'].rolling(window=168, min_periods=1).mean()
df['cnt_roll_std_24']   = df['cnt'].rolling(window=24,  min_periods=1).std().fillna(0)
df['cnt_roll_std_168']  = df['cnt'].rolling(window=168, min_periods=1).std().fillna(0)
df['cnt_roll_max_24']   = df['cnt'].rolling(window=24,  min_periods=1).max()
df['cnt_roll_min_24']   = df['cnt'].rolling(window=24,  min_periods=1).min()

df['temp_x_hum'] = df['temp'] * df['hum']

df['cnt_log'] = np.log1p(df['cnt'])

df.drop(columns=['dteday', 'yr', 'mnth', 'hr', 'weekday', 'cnt'], inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Shape final:", df.shape)
print("Total features:", len(df.columns) - 1)  # -1 por el target
print("Columnas:", df.columns.tolist())

## Construcción de ventanas deslizantes

In [ ]:
TARGET = 'cnt_log'
FEATURES = [c for c in df.columns if c != TARGET]
WINDOW_SIZE = 72

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"Ventana: {WINDOW_SIZE} horas")

In [ ]:
n_total = len(df)
i_train_split = int(n_total * 0.70)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

scaler_X.fit(df[FEATURES].values[:i_train_split])
scaler_y.fit(df[[TARGET]].values[:i_train_split])

X_scaled = scaler_X.transform(df[FEATURES].values)
y_scaled = scaler_y.transform(df[[TARGET]].values).flatten()

def crear_ventanas(X, y, window):
    X_out, y_out = [], []
    for i in range(len(X) - window):
        X_out.append(X[i : i + window])
        y_out.append(y[i + window])
    return np.array(X_out, dtype=np.float32), np.array(y_out, dtype=np.float32)

X_win, y_win = crear_ventanas(X_scaled, y_scaled, WINDOW_SIZE)
print(f"Ventanas: X={X_win.shape}, y={y_win.shape}")

In [ ]:
n = len(X_win)
i_train = int(n * 0.70)
i_val   = int(n * 0.85)

X_train, y_train = X_win[:i_train],       y_win[:i_train]
X_val,   y_val   = X_win[i_train:i_val],  y_win[i_train:i_val]
X_test,  y_test  = X_win[i_val:],         y_win[i_val:]

print(f"Train: {X_train.shape[0]:,} | Val: {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")

## Arquitectura Híbrida: CNN 1D + LSTM Bidireccional

En lugar de una LSTM pura, se propone una arquitectura híbrida que combina:

### 1. Redes Convolucionales 1D (CNN)
Aunque las CNN son famosas por imágenes, en series temporales actúan como **extractores de patrones locales**.
* Los filtros deslizan sobre la ventana detectando "formas" características (picos de consumo, mesetas de inactividad).
* Son invariantes a la posición y reducen el ruido de alta frecuencia antes de pasar los datos a la LSTM.
* Se usa solo **un MaxPooling** (72 → 36) para no perder demasiada resolución temporal.

### 2. LSTM Bidireccional
Recibe las características extraídas por la CNN.
* Al ser bidireccional, analiza la secuencia hacia adelante y hacia atrás, capturando dependencias temporales complejas.

### 3. Flujo de datos
```
Entrada (Ventana 72h × 27 features)
   ↓
[Conv1D(128) → MaxPool(2) → Conv1D(64)]  ← Extracción de patrones locales
   ↓
[BiLSTM(64) → LSTM(32)]                  ← Modelado de secuencia temporal
   ↓
[Dense(64) → Dense(32) → Dense(1)]      ← Predicción
```

In [ ]:
def crear_modelo_cnn_lstm(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),

        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.LayerNormalization(),
        layers.Dropout(0.2),

        layers.LSTM(32),
        layers.LayerNormalization(),
        layers.Dropout(0.2),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),

        layers.Dense(1)
    ])
    return model

In [ ]:
modelo = crear_modelo_cnn_lstm(X_train.shape[1:])
modelo.summary()

## Compilación y Entrenamiento

In [ ]:
modelo.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    EarlyStopping(monitor='val_loss', mode='min', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-7, verbose=1)
]

In [ ]:
historial = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(historial.history['loss'], label='Train Loss')
axes[0].plot(historial.history['val_loss'], label='Val Loss')
axes[0].set_title('Pérdida (MSE)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(historial.history['mae'], label='Train MAE')
axes[1].plot(historial.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Evaluación en Test

In [ ]:
y_pred_scaled = modelo.predict(X_test).flatten()
y_pred_log = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_log = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_true_log)

In [ ]:
mae  = mean_absolute_error(y_true, y_pred)
mse  = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_true, y_pred)

print("=== Métricas (CNN-LSTM + Rolling Features) ===")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(y_true, y_pred, alpha=0.3, s=10, color='steelblue')
axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', linewidth=2)
axes[0].set_title('Predicción vs Real')

axes[1].plot(y_true[:200], label='Real', linewidth=1.5)
axes[1].plot(y_pred[:200], label='Predicción', linewidth=1.5, alpha=0.8)
axes[1].set_title('Comparación (200 muestras)')
axes[1].legend()
plt.tight_layout()
plt.show()